In [1]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torch.optim import Adam
from torch.amp import GradScaler, autocast
from datasets import load_from_disk
import sys
import os
from pathlib import Path
import json

PROJECT_ROOT = Path().resolve().parent
SRC_PATH = PROJECT_ROOT / "src"
MODELS_PATH = PROJECT_ROOT / "models"
BEST_MODEL_SAVED = MODELS_PATH / "resnet50_best.pt"
DATA_PROCESSED_PATH = SRC_PATH / "data" / "processed"
TRAIN_DATASET = DATA_PROCESSED_PATH / "train_cats_dogs"
VALIDATION_DATASET = DATA_PROCESSED_PATH / "validation_cats_dogs"
JSON_PATH = MODELS_PATH / "label_mapping.json"

with open(str(JSON_PATH), "r") as f:
    labels = json.load(f)

EPOCHS = 5
LABEL_MAPPING = labels["labels"]

sys.path.append(str(SRC_PATH))

from data.dataset import CatsDogsDataset
from data.transforms import get_train_transforms, get_val_transformation
from model.resnet import build_resnet50
from model.train import train_one_epoch, validate

## **Carga del dataset**

In [2]:
train_dataset = load_from_disk(str(TRAIN_DATASET))
val_dataset = load_from_disk(str(VALIDATION_DATASET))

train_dataset_pytorch = CatsDogsDataset(train_dataset, transform=get_train_transforms())
val_dataset_pytorch = CatsDogsDataset(val_dataset, transform=get_val_transformation())


## **Entrenamiento del modelo**

In [3]:
device = "cuda" if torch.cuda.is_available() else "cpu"

model = build_resnet50(num_classes=2)
model.to(device)

for param in model.parameters():
    param.requires_grad = False

for param in model.fc.parameters():
    param.requires_grad = True

criterion = torch.nn.CrossEntropyLoss()
optimizer = Adam(model.fc.parameters(), lr=1e-4)

train_loader = DataLoader(train_dataset_pytorch,
                          batch_size=128, 
                          shuffle=True,
                          )

val_loader = DataLoader(val_dataset_pytorch, 
                        batch_size=128,
                        shuffle=False,
                        )

best_val_acc = 0.0

for epoch in range(EPOCHS):
    train_loss, train_acc = train_one_epoch(model, train_loader, optimizer, criterion, device)
    val_loss, val_acc = validate(model, val_loader, criterion, device)

    print(
        f"Epoch {epoch+1}/{EPOCHS} | "
        f"Train Acc: {train_acc:.3f} | "
        f"Val Acc: {val_acc:.3f}"
    )

    if val_acc > best_val_acc:
        best_val_acc = val_acc

        checkpoint = {
            "model_state": model.state_dict(),
            "architecture": "resnet50",
            "num_classes": 2,
            "label_mapping": LABEL_MAPPING,
            "val_accuracy": best_val_acc
        }

        torch.save(checkpoint, BEST_MODEL_SAVED)

        print(f"Nuevo modelo guardado: (val_acc={best_val_acc:.3f})")



Epoch 1/5 | Train Acc: 0.890 | Val Acc: 0.975
Nuevo modelo guardado: (val_acc=0.975)


Epoch 2/5 | Train Acc: 0.967 | Val Acc: 0.979
Nuevo modelo guardado: (val_acc=0.979)


Epoch 3/5 | Train Acc: 0.971 | Val Acc: 0.981
Nuevo modelo guardado: (val_acc=0.981)


Epoch 4/5 | Train Acc: 0.974 | Val Acc: 0.980


Epoch 5/5 | Train Acc: 0.975 | Val Acc: 0.982
Nuevo modelo guardado: (val_acc=0.982)
